In [1]:
import wrds

conn = wrds.Connection()


d:\Documents\School\cfm301\project1\.venv\Lib\site-packages\wrds\sql.py:229: SyntaxWarning: invalid escape sequence '\p'
  and is stored in the %APPDATA%\postgresql directory.
d:\Documents\School\cfm301\project1\.venv\Lib\site-packages\wrds\sql.py:249: SyntaxWarning: invalid escape sequence '\p'
  Save the pgpass file in %APPDATA%\postgresql as 'pgpass.conf'.
d:\Documents\School\cfm301\project1\.venv\Lib\site-packages\wrds\sql.py:293: SyntaxWarning: invalid escape sequence '\:'
  passwd = passwd.replace(":", "\:")
d:\Documents\School\cfm301\project1\.venv\Lib\site-packages\wrds\sql.py:304: SyntaxWarning: invalid escape sequence '\:'
  oldline = line.replace("""\:""", "##COLON##")


OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: fe_sendauth: no password supplied

(Background on this error at: https://sqlalche.me/e/14/e3q8)

In [1]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

import pandas as pd

D:\Documents\School\cfm301\project1


In [2]:
start_date = "1980-01-01"
end_date   = "2026-09-30"

ccm_cte = f"""
ccm as (
    select
        ugvkey     as gvkey,
        uiid       as liid,
        apermno    as permno,
        upermco    as permco,
        ulinkdt    as linkdt,
        ulinkenddt as linkenddt,
        ulinktype  as linktype,
        ulinkprim  as linkprim
    from crsp_a_ccm.ccmxpf_lnkused
    where usedflag = 1
      and ulinktype in ('LC','LU','LS')
      and ulinkprim in ('P','C')
      and ulinkdt <= date '{end_date}'
      and coalesce(ulinkenddt, date '9999-12-31') >= date '{start_date}'
)
"""

In [4]:
sql_links = f"""
with {ccm_cte}
select *
from ccm
order by permno, linkdt, linkenddt
"""

links_df = conn.raw_sql(sql_links)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed


# Conservative defaults for WRDS.
DAILY_MAX_WORKERS = 4
DAILY_CHUNK_YEARS = 5


def make_daily_chunks(start_date, end_date, years_per_chunk):
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    chunks = []
    chunk_start = start

    while chunk_start <= end:
        chunk_end = min(
            chunk_start
            + pd.DateOffset(years=years_per_chunk)
            - pd.Timedelta(days=1),
            end,
        )

        chunks.append(
            (
                chunk_start.strftime("%Y-%m-%d"),
                chunk_end.strftime("%Y-%m-%d"),
            )
        )

        chunk_start = chunk_end + pd.Timedelta(days=1)

    return chunks


def build_daily_sql(chunk_start, chunk_end):
    return f"""
with {ccm_cte},

names as (

    select

        permno,

        namedt,

        nameendt,

        exchcd,

        shrcd,

        comnam

    from crsp_a_stock.dsenames

    where exchcd in (1,2,3)

      and shrcd in (10,11)

),

daily as (

    select

        d.*,

        n.exchcd as name_exchcd,

        n.shrcd as name_shrcd,

        n.namedt as name_namedt,

        n.nameendt as name_nameendt,

        n.comnam as company_name,

        c.gvkey as ccm_gvkey,

        c.liid as ccm_liid,

        c.permno as ccm_permno,

        c.permco as ccm_permco,

        c.linkdt as ccm_linkdt,

        c.linkenddt as ccm_linkenddt,

        c.linktype as ccm_linktype,

        c.linkprim as ccm_linkprim,

        row_number() over (

            partition by d.permno, d.date

            order by

                case when c.linkprim = 'P' then 1 else 2 end,

                case when c.linktype = 'LC' then 1

                     when c.linktype = 'LU' then 2

                     when c.linktype = 'LS' then 3

                     else 9 end,

                c.linkdt desc

        ) as rn

    from crsp_a_stock.dsf d

    join names n

      on d.permno = n.permno

     and d.date between n.namedt and n.nameendt

    join ccm c

      on d.permno = c.permno

     and d.date between c.linkdt
                    and coalesce(c.linkenddt, date '9999-12-31')

    where d.date between date '{chunk_start}'
                     and date '{chunk_end}'

)

select *

from daily

where rn = 1

order by permno, date
"""


WRDS_USERNAME = (
    getattr(conn, "wrds_username", None)
    or getattr(conn, "_username", None)
    or os.getenv("WRDS_USERNAME")
)

if not WRDS_USERNAME:
    WRDS_USERNAME = input("WRDS username: ").strip()


def pull_daily_chunk(chunk_start, chunk_end):
    # Each thread receives its own connection.
    worker_conn = wrds.Connection(
        wrds_username=WRDS_USERNAME
    )

    try:
        chunk_df = (
            worker_conn
            .raw_sql(build_daily_sql(chunk_start, chunk_end))
            .drop(columns=["rn"], errors="ignore")
        )

        return chunk_start, chunk_end, chunk_df

    finally:
        worker_conn.close()


daily_chunks = make_daily_chunks(
    start_date=start_date,
    end_date=end_date,
    years_per_chunk=DAILY_CHUNK_YEARS,
)

daily_results = []

with ThreadPoolExecutor(
    max_workers=min(DAILY_MAX_WORKERS, len(daily_chunks))
) as executor:

    future_to_chunk = {
        executor.submit(
            pull_daily_chunk,
            chunk_start,
            chunk_end,
        ): (chunk_start, chunk_end)
        for chunk_start, chunk_end in daily_chunks
    }

    for future in as_completed(future_to_chunk):
        chunk_start, chunk_end = future_to_chunk[future]

        try:
            _, _, chunk_df = future.result()

        except Exception as error:
            for pending_future in future_to_chunk:
                pending_future.cancel()

            raise RuntimeError(
                f"Daily-price pull failed for "
                f"{chunk_start} through {chunk_end}"
            ) from error

        daily_results.append(chunk_df)

        print(
            f"Completed daily prices: "
            f"{chunk_start} through {chunk_end} "
            f"({len(chunk_df):,} rows)"
        )


# Futures finish in an arbitrary order. This restores the exact global
# ordering imposed by the original SQL query.
daily_df = (
    pd.concat(daily_results, ignore_index=True)
      .sort_values(
          ["permno", "date"],
          kind="stable",
      )
      .reset_index(drop=True)
)

print(f"Complete daily dataset: {len(daily_df):,} rows")

Loading library list...
Loading library list...
Loading library list...
Loading library list...
Done
Done
Done
Done


**Note on `company_name`:** the query above now also pulls `comnam` from `crsp.dsenames`
(aliased `company_name`) so downstream notebooks can label stocks by name instead of only
CUSIP/PERMNO. This was added after the data in this submission was already pulled, so the
current `data/raw/daily_df.parquet` does not yet have this column -- it will appear the
next time this notebook is actually run against WRDS. `02_clean/assign1_clean.ipynb` will
need `"company_name"` added to its `daily_cols` list (and a pass-through rename) to carry
it forward once that re-pull happens.

#### Delisted stock returns


In [5]:
sql = """
SELECT
    permno,
    dlstdt,
    dlstcd,
    dlret,
    dlretx,
    dlprc,
    dlamt,
    nextdt,
    dlpdt,
    nwperm,
    nwcomp,
    acperm,
    accomp
FROM crsp_a_stock.dsedelist
WHERE dlstdt BETWEEN %s AND %s
ORDER BY permno, dlstdt
"""

dlist = conn.raw_sql(sql, params=(start_date, end_date))

In [ ]:
dlist.to_parquet(RAW_DIR / "dlist_df.parquet", index=False)
links_df.to_parquet(RAW_DIR / "links_df.parquet", index=False)
daily_df.to_parquet(RAW_DIR / "daily_df.parquet", index=False)
conn.close()